# Exercise 5 — Temporal Concept Analysis (optional, 2 points)

This notebook is fully self-contained and does not modify any file from
Exercises 1-4 — it only *reads* `data/wine_sample_200.csv` and *imports*
(does not edit) the scale-building functions from `ex2_prepare_context.py`.

**The honest framing, stated up front**: the wine dataset has no real time
dimension (no vintage, no repeated measurements of the same wine). As a fallback, quality tier (Low -> Medium -> High) is used as
a **simulated time axis** ("stages of development"), not real chronological
time. This is a deliberate simplification, not a claim that wines actually
transition between tiers.

A bigger consequence of that choice, already proven in Exercise 4: no wine
belongs to more than one tier (Y is a strict object partition). So no
individual wine can have a life track in the classical sense — there is no
persisting object to trace across stages. The adaptation made here: track
**concepts** (attribute combinations) across the tier sequence instead of
individual objects, and define life-track events (birth, death, refinement,
split, merge, stability) for concepts rather than for objects. This is
flagged explicitly rather than glossed over, exactly as Exercise 4 flagged
the degenerate condition-implication family.

In [1]:
import pandas as pd
import sys
sys.path.insert(0, ".")
from ex2_prepare_context import build_derived_context, all_attributes
import concepts

sample = pd.read_csv("../data/wine_sample_200.csv")
derived = build_derived_context(sample)
attrs = all_attributes()
chem_attrs = [a for a in attrs if not a.startswith("quality")]
tiers = sample.set_index("wine_id")["quality_tier"].to_dict()
obj_attrs = {wid: set(a for a in derived[wid] if a in chem_attrs) for wid in derived}

print(f"{len(chem_attrs)} chemistry attributes, time axis = Low -> Medium -> High")


30 chemistry attributes, time axis = Low -> Medium -> High


## 1. Build the three time-slice contexts

Recomputed independently here (not reusing `ex4_triadic.ipynb`'s in-memory
state, to keep this notebook self-contained) but over the same data, so the
lattice sizes should match Exercise 4 exactly — a useful cross-check.

In [2]:
tier_wids = {t: [w for w, tt in tiers.items() if tt == t] for t in ["Low", "Medium", "High"]}
lattices = {}
for tier, wids in tier_wids.items():
    objects = [str(w) for w in wids]
    bools = [tuple(a in obj_attrs[w] for a in chem_attrs) for w in wids]
    ctx = concepts.Context(objects, chem_attrs, bools)
    lattices[tier] = ctx.lattice
    print(f"t={tier}: {len(wids)} wines -> {len(ctx.lattice)} concepts")


t=Low: 8 wines -> 43 concepts


t=Medium: 165 wines -> 4781 concepts
t=High: 27 wines -> 599 concepts


## 2. Matching rule between consecutive stages

Since extents can't be compared directly (no shared objects between
stages), concepts are matched by **intent** (attribute combination) using
set inclusion:

- a concept `I` in stage A is matched to its **minimal supersets** in stage B
  (the closest, most direct continuations — not *all* supersets, which would
  explode combinatorially in the much larger Medium lattice)
- **stable**: exactly one minimal superset, and it equals `I`
- **refined**: exactly one minimal superset, strictly larger than `I`
  (the combination persists but gains attributes)
- **split**: 2+ minimal supersets (the combination forks into multiple
  more specific successors)
- **died**: zero supersets at all (the combination has no continuation)
- **born** (computed from stage B's side): a stage-B intent with no subset
  at all among stage-A intents — a genuinely new combination

Only concepts with **nonempty extent** (real wines behind them, not the
vacuous all-attributes bottom concept that every formal context has by
construction) are used — the vacuous bottom concept would trivially "match"
everything and add no information.

In [3]:
def minimal_supersets(I, candidates):
    sups = [J for J in candidates if I <= J]
    return [J for J in sups if not any(K < J for K in sups if K != J)]


real_intents = {
    t: [frozenset(c.intent) for c in lattices[t] if len(c.extent) > 0]
    for t in lattices
}
print({t: len(real_intents[t]) for t in real_intents}, "real (nonempty-extent) concepts per stage")


def classify_transition(stageA, stageB):
    intentsA, intentsB = real_intents[stageA], real_intents[stageB]
    events = {"stable": 0, "refined": 0, "split": 0, "died": 0}
    details = {"stable": [], "refined": [], "split": [], "died": []}
    for I in intentsA:
        sups = minimal_supersets(I, intentsB)
        if not sups:
            events["died"] += 1
            details["died"].append(I)
        elif len(sups) == 1 and sups[0] == I:
            events["stable"] += 1
            details["stable"].append(I)
        elif len(sups) == 1:
            events["refined"] += 1
            details["refined"].append((I, sups[0]))
        else:
            events["split"] += 1
            details["split"].append((I, sups))
    born = [J for J in intentsB if not any(I <= J for I in intentsA)]
    events["born"] = len(born)
    details["born"] = born
    return events, details


events_lm, details_lm = classify_transition("Low", "Medium")
events_mh, details_mh = classify_transition("Medium", "High")
print("Low -> Medium:", events_lm)
print("Medium -> High:", events_mh)


{'Low': 42, 'Medium': 4780, 'High': 598} real (nonempty-extent) concepts per stage


Low -> Medium:

 {'stable': 23, 'refined': 13, 'split': 0, 'died': 6, 'born': 3666}
Medium -> High: {'stable': 364, 'refined': 2681, 'split': 0, 'died': 1735, 'born': 0}


## 3. Validating the matching logic on a toy case

Before trusting "zero splits found" as a real finding rather than a bug in
`minimal_supersets`, the function is checked against two hand-built cases
where the right answer is known.

In [4]:
# toy split case: {a} should match both {a,b} and {a,c} (genuine fork)
toy_split = minimal_supersets(frozenset({"a"}), [frozenset({"a", "b"}), frozenset({"a", "c"})])
print("toy split case (expect 2 matches):", toy_split)

# toy chain case: {a} should match only {a,b}, not the larger {a,b,c} (not minimal)
toy_chain = minimal_supersets(frozenset({"a"}), [frozenset({"a", "b"}), frozenset({"a", "b", "c"})])
print("toy chain case (expect 1 match):", toy_chain)


toy split case (expect 2 matches): [frozenset({'a', 'b'}), frozenset({'a', 'c'})]
toy chain case (expect 1 match): [frozenset({'a', 'b'})]


Both toy cases behave correctly, so the **zero splits found in the real
data** (both transitions) is a genuine structural property of this context,
not a logic bug: every surviving concept refines along a single deterministic
path rather than forking into incompatible specializations. This is specific
to this binary/ordinal-scaled wine context, not a general law of TCA.

## 4. Concrete examples of each event type

In [5]:
print("DIED (Low -> Medium) — a combination with no continuation at all:")
print(" ", sorted(details_lm["died"][0]))

print("\nREFINED (Low -> Medium) — persists but gains an attribute:")
before, after = details_lm["refined"][0]
print(" before:", sorted(before))
print(" after: ", sorted(after))
print(" added: ", sorted(after - before))

print("\nSTABLE (Low -> Medium) — identical combination in both stages:")
print(" ", sorted(details_lm["stable"][0]))

print("\nDIED (Medium -> High):")
print(" ", sorted(details_mh["died"][0]))


DIED (Low -> Medium) — a combination with no continuation at all:
  ['alcohol>=9', 'chlorides=high_salt', 'citric_acid_absent', 'density=heavy', 'fixed_acidity>=low(6)', 'pH>=basic(3.4)', 'residual_sugar=off_dry', 'sulphates>=low(0.4)', 'volatile_acidity>=high(0.7)', 'volatile_acidity>=low(0.3)', 'volatile_acidity>=medium(0.5)']

REFINED (Low -> Medium) — persists but gains an attribute:
 before: ['alcohol>=9', 'chlorides=low_salt', 'citric_acid_absent', 'density=light', 'fixed_acidity>=low(6)', 'free_so2>=low(6)', 'pH>=basic(3.4)', 'residual_sugar=dry', 'sulphates>=low(0.4)', 'sulphates>=medium(0.6)', 'volatile_acidity>=high(0.7)', 'volatile_acidity>=low(0.3)', 'volatile_acidity>=medium(0.5)']
 after:  ['alcohol>=11', 'alcohol>=9', 'chlorides=low_salt', 'citric_acid_absent', 'density=light', 'fixed_acidity>=low(6)', 'free_so2>=low(6)', 'pH>=basic(3.4)', 'residual_sugar=dry', 'sulphates>=low(0.4)', 'sulphates>=medium(0.6)', 'volatile_acidity>=high(0.7)', 'volatile_acidity>=low(0.3)', '

## 5. Summary table

In [6]:
summary = pd.DataFrame([events_lm, events_mh], index=["Low -> Medium", "Medium -> High"])
summary


,stable,refined,split,died,born
Low -> Medium,23,13,0,6,3666
Medium -> High,364,2681,0,1735,0
